In [ ]:
# ============================================================
# Project: NCR Ride Booking — Data Mining & Predictive Modelling
# Author : Vila Chung
# Purpose: Build a cancellation-prediction classifier, evaluate
#          its performance, and extract actionable feature insights.
# ============================================================

# ── Cell 1: Imports ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, RocCurveDisplay,
)

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
    print("XGBoost detected — will use XGBClassifier.")
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not installed — falling back to RandomForestClassifier.")

sns.set(style="whitegrid")
print("Environment ready.")

In [ ]:
# ── Cell 2: Load Cleaned Data ────────────────────────────────
CLEANED_FILE = "cleaned_ncr_rides.csv"

try:
    df = pd.read_csv(CLEANED_FILE, parse_dates=["Datetime", "Date_only"])
    print(f"Loaded — shape: {df.shape}")
    print("Columns:", df.columns.tolist())
except FileNotFoundError:
    raise FileNotFoundError(
        f"'{CLEANED_FILE}' not found. "
        "Run 01_Data_Cleaning_and_Preparation.ipynb first."
    )

In [ ]:
# ── Cell 3: Define Target Variable ───────────────────────────
# Binary label: 1 = cancelled / failed, 0 = completed
df["is_cancelled"] = (df["Cancel_Type"] != "Completed").astype(int)

print("Class distribution:")
print(df["is_cancelled"].value_counts(normalize=True).mul(100).round(1).to_string())

In [ ]:
# ── Cell 4: Feature Engineering ──────────────────────────────
# Encode categorical features as integer codes
CAT_COLS = ["Vehicle Type", "Pickup Location", "Drop Location", "Payment Method"]
le = LabelEncoder()
for col in CAT_COLS:
    if col in df.columns:
        df[f"{col}_enc"] = le.fit_transform(df[col].astype(str))

# Select feature set (only numeric / encoded columns)
FEATURE_COLS = [
    "Hour", "Weekday", "Is_Weekend", "Month",
    "Ride Distance",
    "Booking Value",
    "Vehicle Type_enc",
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

X = df[FEATURE_COLS].copy()
y = df["is_cancelled"]

print(f"\nFeature set ({len(FEATURE_COLS)} features): {FEATURE_COLS}")

In [ ]:
# ── Cell 5: Train / Test Split & Preprocessing ───────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Impute any remaining NaNs with column medians from training set
train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_test  = X_test.fillna(train_medians)   # use training medians to avoid data leakage

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

In [ ]:
# ── Cell 6: Train Model ───────────────────────────────────────
if XGB_AVAILABLE:
    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1,
    )
    model_name = "XGBoost"
else:
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced",   # handles class imbalance automatically
    )
    model_name = "RandomForest"

# Tree-based models work directly on unscaled data
model.fit(X_train, y_train)
print(f"{model_name} trained successfully.")

# 5-fold cross-validation AUC (gives a more reliable performance estimate)
cv_auc = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1)
print(f"5-fold CV ROC-AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")

In [ ]:
# ── Cell 7: Model Evaluation ─────────────────────────────────
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    RocCurveDisplay, PrecisionRecallDisplay
)

y_pred       = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Text Report
print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["Completed", "Cancelled"]))
print(f"Test ROC-AUC : {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"Avg Precision: {average_precision_score(y_test, y_pred_proba):.4f}")

# Three Figures: Confusion Matrix + ROC + PR Curve
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Figure1：Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
    xticklabels=["Completed", "Cancelled"],
    yticklabels=["Completed", "Cancelled"],
)
axes[0].set_title("Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

# Figure2：ROC Curve
RocCurveDisplay.from_predictions(
    y_test, y_pred_proba, ax=axes[1], name=model_name
)
axes[1].set_title(f"ROC Curve (AUC = {roc_auc_score(y_test, y_pred_proba):.4f})")

# Figure3：PR Curve
# Specifically for Imbalanced Data
PrecisionRecallDisplay.from_predictions(
    y_test, y_pred_proba, ax=axes[2], name=model_name
)
ap = average_precision_score(y_test, y_pred_proba)
axes[2].set_title(f"Precision-Recall Curve (AP = {ap:.4f})")

plt.tight_layout()
plt.savefig("03_model_evaluation.png", dpi=150)
plt.show()

In [ ]:
# ── Cell 8: Feature Importance ───────────────────────────────
if hasattr(model, "feature_importances_"):
    importances = (
        pd.Series(model.feature_importances_, index=FEATURE_COLS)
        .sort_values(ascending=False)
    )

    plt.figure(figsize=(10, 6))
    sns.barplot(x=importances.values, y=importances.index, palette="viridis")
    plt.title(f"Feature Importance — {model_name}")
    plt.xlabel("Importance Score")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.savefig("03_feature_importance.png", dpi=150)
    plt.show()

    print("Top 10 features:")
    display(importances.head(10).round(4).to_frame("Importance"))

In [ ]:
# ── Cell 9: Export Dataset with Target Label ──────────────────
OUT_FILE = "cleaned_ncr_rides_with_target.csv"
df.to_csv(OUT_FILE, index=False)
print(f"Saved with 'is_cancelled' column → {OUT_FILE}")

In [ ]:
# ──Cell Added: K-means Order Risk Clustering ─────────────────────────
# K-means is for teaching clustering concepts
# Production usually uses DBSCAN or business rules

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Cluster with two key features
cluster_features = ["Ride Distance", "Booking Value"]
X_cluster = df[cluster_features].dropna()

# Standardize data (K-means is scale-sensitive)
scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(X_cluster)

# Find best K (Elbow Method)
inertias = []
K_range  = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, "bo-")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method — Optimal K")
plt.tight_layout()
plt.savefig("03_elbow.png", dpi=150)
plt.show()

# Use K=3 (low/medium/high risk orders)
km_final = KMeans(n_clusters=3, random_state=42, n_init=10)
df.loc[X_cluster.index, "Risk_Cluster"] = km_final.fit_predict(X_scaled)

# Check cancellation rate per cluster
cluster_summary = (
    df.groupby("Risk_Cluster")
    .agg(
        Count=("is_cancelled", "size"),
        Cancel_Rate=("is_cancelled", "mean"),
        Avg_Distance=("Ride Distance", "mean"),
        Avg_Value=("Booking Value", "mean"),
    )
    .assign(Cancel_Rate=lambda x: (x["Cancel_Rate"]*100).round(1))
    .round(1)
)
print("=== Risk Cluster Summary ===")
display(cluster_summary)